In [4]:
# 1) BASIC SETUP
from transformers import DataCollatorForTokenClassification

from google.colab import drive
drive.mount('/content/drive')

import re
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

from datasets import Dataset, DatasetDict  # <-- fixes the NameError

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2) LOAD CLEAN NOTES
CSV_PATH = "/content/drive/MyDrive/diabetes_patient_notes_clean.csv"  # <- your file
dev_df = pd.read_csv(CSV_PATH)

print("dev_df shape:", dev_df.shape)
print("dev_df columns:", dev_df.columns.tolist())
print("\nFirst note preview:\n", dev_df["clean_text"].iloc[0][:500])


Mounted at /content/drive
Using device: cpu
dev_df shape: (18618, 3)
dev_df columns: ['subject_id', 'text', 'clean_text']

First note preview:
 name unit no admission date discharge date date of birth sex f service medicine allergies no known allergies adverse drug reactions attending chief complaint shortness of breath major surgical or invasive procedure none history of present illness yo woman with h o hypertension hyperlipidemia diabetes mellitus on insulin therapy h o cerebellar medullary stroke in ckd stage iii iv presenting with fatigue and dyspnea on exertion doe for a few weeks markedly worse this morning over the past few week


In [6]:
# 3) LABEL SCHEMA
LABELS = [
    "O",
    "B-LABTEST", "I-LABTEST",
    "B-LABVALUE", "I-LABVALUE",
    "B-UNIT", "I-UNIT",
    "B-DATE", "I-DATE"
]
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
print("Labels:", LABELS)
print("label2id:", label2id)


Labels: ['O', 'B-LABTEST', 'I-LABTEST', 'B-LABVALUE', 'I-LABVALUE', 'B-UNIT', 'I-UNIT', 'B-DATE', 'I-DATE']
label2id: {'O': 0, 'B-LABTEST': 1, 'I-LABTEST': 2, 'B-LABVALUE': 3, 'I-LABVALUE': 4, 'B-UNIT': 5, 'I-UNIT': 6, 'B-DATE': 7, 'I-DATE': 8}


In [7]:
# 4) REGEX + SIMPLE TOKENIZER

RE_VALUE = re.compile(r"^\d+(\.\d+)?%?$")  # 7, 7.5, 7.5%
RE_DATE  = re.compile(
    r"^(\d{1,2}/\d{1,2}/\d{2,4}|\d{4}-\d{1,2}-\d{1,2}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)$",
    re.IGNORECASE,
)
UNITS = {"mg/dl", "mmol/l", "g/dl", "%"}
LAB_KEYWORDS = {
    "hba1c", "a1c", "glucose", "blood", "sugar",
    "creatinine", "gfr", "egfr", "bun"
}

def simple_tokenize(text: str):
    return str(text).strip().replace("\n", " ").split()

def weak_label_tokens(tokens):
    labels = []
    prev_type = None

    for tok in tokens:
        low = tok.lower()

        # LABTEST
        if low in LAB_KEYWORDS:
            cur_type = "LABTEST"

        # VALUE
        elif RE_VALUE.match(low):
            cur_type = "LABVALUE"

        # UNIT
        elif low in UNITS:
            cur_type = "UNIT"

        # DATE
        elif RE_DATE.match(low):
            cur_type = "DATE"

        else:
            cur_type = None

        if cur_type is None:
            labels.append(label2id["O"])
            prev_type = None
        else:
            prefix = "B" if prev_type != cur_type else "I"
            label_name = f"{prefix}-{cur_type}"
            labels.append(label2id[label_name])
            prev_type = cur_type

    return labels


In [8]:
sample_text = dev_df["clean_text"].iloc[0]
tokens = simple_tokenize(sample_text[:400])
labels = weak_label_tokens(tokens)

for t, l in zip(tokens[:50], labels[:50]):
    print(f"{t:15} -> {LABELS[l]}")


name            -> O
unit            -> O
no              -> O
admission       -> O
date            -> O
discharge       -> O
date            -> O
date            -> O
of              -> O
birth           -> O
sex             -> O
f               -> O
service         -> O
medicine        -> O
allergies       -> O
no              -> O
known           -> O
allergies       -> O
adverse         -> O
drug            -> O
reactions       -> O
attending       -> O
chief           -> O
complaint       -> O
shortness       -> O
of              -> O
breath          -> O
major           -> O
surgical        -> O
or              -> O
invasive        -> O
procedure       -> O
none            -> O
history         -> O
of              -> O
present         -> O
illness         -> O
yo              -> O
woman           -> O
with            -> O
h               -> O
o               -> O
hypertension    -> O
hyperlipidemia  -> O
diabetes        -> O
mellitus        -> O
on              -> O
insulin      

In [9]:
MAX_NOTES = 2000    # keep this small first
MAX_TOKENS = 256    # truncate long notes

tokens_list = []
labels_list = []

for text in dev_df["clean_text"].fillna("").iloc[:MAX_NOTES]:
    toks = simple_tokenize(text)[:MAX_TOKENS]
    labs = weak_label_tokens(toks)
    # safety
    if len(toks) != len(labs):
        continue
    tokens_list.append(toks)
    labels_list.append(labs)

df_ner = pd.DataFrame({"tokens": tokens_list, "labels": labels_list})
print("df_ner shape:", df_ner.shape)
df_ner.head()


df_ner shape: (2000, 2)


,tokens,labels
0,"[name, unit, no, admission, date, discharge, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[name, unit, no, admission, date, discharge, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[name, unit, no, admission, date, discharge, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[name, unit, no, admission, date, discharge, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[name, unit, no, admission, date, discharge, d...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [6]:
save_path = "/content/drive/MyDrive/df_ner_option3.pkl"
df_ner.to_pickle(save_path)
print("Saved df_ner to:", save_path)


Saved df_ner to: /content/drive/MyDrive/df_ner_option3.pkl


In [10]:
from datasets import Dataset, DatasetDict  # already imported above

raw_dataset = Dataset.from_pandas(df_ner)

splits = raw_dataset.train_test_split(test_size=0.2, seed=42)
dataset = DatasetDict({
    "train": splits["train"],
    "validation": splits["test"],
})

dataset


DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 400
    })
})

In [11]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
    )

    all_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=i) for i in range(len(examples["tokens"]))):
        word_labels = examples["labels"][i]
        prev_word = None
        label_ids = []
        for w_id in word_ids:
            if w_id is None:
                label_ids.append(-100)
            elif w_id != prev_word:
                label_ids.append(word_labels[w_id])
                prev_word = w_id
            else:
                # same word split into sub-tokens → use same label
                label_ids.append(word_labels[w_id])
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)

    mask = labels != -100
    correct = (preds[mask] == labels[mask]).sum()
    total = mask.sum()
    acc = (correct / total).item() if total > 0 else 0.0
    return {"accuracy": acc}


In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/clinbert_lab_ner_option3",
    num_train_epochs=1,              # keep it small for now
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    logging_steps=50,
    save_steps=200,
    fp16=False,                      # CPU only
    # DO NOT pass evaluation_strategy here (your version doesn't support it)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],  # still okay; we'll call evaluate() explicitly
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,          # or None if you want it even simpler
)

trainer.train()


/tmp/ipython-input-4285186836.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bbmukumb (bbmukumb-michigan-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.377400
100,0.029500
150,0.012600
200,0.004500
250,0.004700
300,0.003200
350,0.002700
400,0.002600


TrainOutput(global_step=400, training_loss=0.05463812485337258, metrics={'train_runtime': 4510.6588, 'train_samples_per_second': 0.355, 'train_steps_per_second': 0.089, 'total_flos': 209050634649600.0, 'train_loss': 0.05463812485337258, 'epoch': 1.0})

In [14]:
eval_results = trainer.evaluate()
print(eval_results)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.002834758721292019, 'eval_accuracy': 0.9996259842519685, 'eval_runtime': 251.5136, 'eval_samples_per_second': 1.59, 'eval_steps_per_second': 0.398, 'epoch': 1.0}


In [15]:
# 1) Where to save
save_dir = "/content/drive/MyDrive/clinbert_lab_ner_option3"

# 2) Save model + tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model + tokenizer to:", save_dir)


Saved model + tokenizer to: /content/drive/MyDrive/clinbert_lab_ner_option3


In [16]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

load_dir = "/content/drive/MyDrive/clinbert_lab_ner_option3"
tokenizer = AutoTokenizer.from_pretrained(load_dir)
model = AutoModelForTokenClassification.from_pretrained(load_dir).to("cpu")


In [15]:
import torch

# Make sure label2id exists from earlier; if not, recreate:
# label2id = {'O': 0, 'B-LABTEST': 1, 'I-LABTEST': 2,
#             'B-LABVALUE': 3, 'I-LABVALUE': 4,
#             'B-UNIT': 5, 'I-UNIT': 6,
#             'B-DATE': 7, 'I-DATE': 8}

id2label = {v: k for k, v in label2id.items()}

def predict_tokens_and_labels(text, max_len=256):
    """
    Run the trained model on a single note string and return:
      - wordpiece tokens
      - predicted BIO labels
    """
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding=False,
    )

    with torch.no_grad():
        outputs = model(**enc)
        logits = outputs.logits  # [1, seq_len, num_labels]
        preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    input_ids = enc["input_ids"][0].cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    labels = [id2label[p] for p in preds]

    return tokens, labels


In [16]:
def merge_subword(token):
    # remove WordPiece "##"
    return token.replace("##", "")

def get_entity_spans(tokens, labels):
    """
    Convert BIO labels into spans:
      [(ent_type, surface_form), ...]
    """
    spans = []
    current_type = None
    current_tokens = []

    for tok, lab in zip(tokens, labels):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        if lab == "O":
            if current_type is not None:
                spans.append((current_type, " ".join(current_tokens)))
                current_type, current_tokens = None, []
            continue

        # split B-LABTEST -> B, LABTEST
        prefix, ent_type = lab.split("-", 1)

        if prefix == "B":
            # close old span
            if current_type is not None:
                spans.append((current_type, " ".join(current_tokens)))
            current_type = ent_type
            current_tokens = [merge_subword(tok)]
        else:  # "I"
            if current_type == ent_type:
                current_tokens.append(merge_subword(tok))
            else:
                # incorrect I- start, treat as new B-
                if current_type is not None:
                    spans.append((current_type, " ".join(current_tokens)))
                current_type = ent_type
                current_tokens = [merge_subword(tok)]

    if current_type is not None:
        spans.append((current_type, " ".join(current_tokens)))

    return spans


In [17]:
# Quick sanity check on first 3 notes
for i in range(3):
    txt = str(dev_df["clean_text"].iloc[i])
    print("="*70)
    print(f"=== NOTE {i} ===")
    print("TEXT PREVIEW:", txt[:400].replace("\n", " "), "...\n")

    tokens, labels = predict_tokens_and_labels(txt)
    spans = get_entity_spans(tokens, labels)

    if not spans:
        print("SPANS: (no entities found)\n")
    else:
        print("SPANS:")
        for ent_type, surface in spans:
            print(f"  {ent_type:9} -> {surface}")
    print()


=== NOTE 0 ===
TEXT PREVIEW: name unit no admission date discharge date date of birth sex f service medicine allergies no known allergies adverse drug reactions attending chief complaint shortness of breath major surgical or invasive procedure none history of present illness yo woman with h o hypertension hyperlipidemia diabetes mellitus on insulin therapy h o cerebellar medullary stroke in ckd stage iii iv presenting with fa ...

SPANS:
  LABVALUE  -> 1
  LABVALUE  -> 2
  LABVALUE  -> 3

=== NOTE 1 ===
TEXT PREVIEW: name unit no admission date discharge date date of birth sex f service medicine allergies lisinopril attending chief complaint back pain major surgical or invasive procedure none history of present illness the patient is a y o f with pmhx significant for htn gerd cad s p cabg and stenting iddm with periperal neuropathy who presents with r flank pain per patient this pain has been going on for the  ...

SPANS:
  LABVALUE  -> 3
  LABVALUE  -> 2
  LABVALUE  -> 4
  LABVALUE  -

In [18]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

# ---- reproducibility ----
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# ---- disable W&B (no crashes, no login) ----
os.environ["WANDB_DISABLED"] = "true"

# ============================================
# 1) LOAD df_ner (tokens + labels)
#    Either: read from CSV, or reuse your df_ner variable
# ============================================

# If you saved df_ner:
# df_ner = pd.read_csv("/content/drive/MyDrive/diabetes_nlp/df_ner.csv")

# If df_ner is already in memory, just make sure it has:
#   columns: ['tokens', 'labels']
print("df_ner shape:", df_ner.shape)
print(df_ner.head())


df_ner shape: (2000, 2)
                                              tokens  \
0  [name, unit, no, admission, date, discharge, d...   
1  [name, unit, no, admission, date, discharge, d...   
2  [name, unit, no, admission, date, discharge, d...   
3  [name, unit, no, admission, date, discharge, d...   
4  [name, unit, no, admission, date, discharge, d...   

                                              labels  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  


In [19]:
# ============================================
# 2) Label schema (BIO)
# ============================================
labels = [
    "O",
    "B-LABTEST",
    "I-LABTEST",
    "B-LABVALUE",
    "I-LABVALUE",
    "B-UNIT",
    "I-UNIT",
    "B-DATE",
    "I-DATE",
]

label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}

print("Labels:", labels)
print("label2id:", label2id)

# ============================================
# 3) Convert df_ner -> HuggingFace Dataset
#    (optionally subsample for RAM safety)
# ============================================

# Optional: keep only first N notes to reduce RAM/time
N = 3000  # you can increase if RAM is fine
df_small = df_ner.iloc[:N].reset_index(drop=True)

raw_dataset = Dataset.from_pandas(df_small)

splits = raw_dataset.train_test_split(test_size=0.2, seed=seed)
dataset = DatasetDict({
    "train": splits["train"],
    "validation": splits["test"],
})

print(dataset)

# ============================================
# 4) Load tokenizer + model
# ============================================
model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


Labels: ['O', 'B-LABTEST', 'I-LABTEST', 'B-LABVALUE', 'I-LABVALUE', 'B-UNIT', 'I-UNIT', 'B-DATE', 'I-DATE']
label2id: {'O': 0, 'B-LABTEST': 1, 'I-LABTEST': 2, 'B-LABVALUE': 3, 'I-LABVALUE': 4, 'B-UNIT': 5, 'I-UNIT': 6, 'B-DATE': 7, 'I-DATE': 8}
DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 400
    })
})


Some weights of BertForTokenClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
# ============================================
# 5) Tokenization with label alignment
#    df_ner["tokens"] is a list of words
#    df_ner["labels"] is a list of ints (same length)
# ============================================

def tokenize_and_align_labels(examples):
    # examples["tokens"] is a list of token-lists
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    all_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=i) for i in range(len(examples["tokens"]))):
        labels_ids = examples["labels"][i]
        aligned = []
        prev_word_id = None
        for w_id in word_ids:
            if w_id is None:
                aligned.append(-100)  # ignore special tokens
            else:
                # only label the first subword, ignore the rest
                if w_id != prev_word_id:
                    aligned.append(labels_ids[w_id])
                else:
                    aligned.append(-100)
            prev_word_id = w_id
        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized_ds)


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 400
    })
})


In [21]:
# ============================================
# 6) Data collator
# ============================================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# ============================================
# 7) Metrics
# ============================================
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(-1)

    # Flatten, ignoring -100
    true_labels = []
    true_preds  = []
    for p_row, l_row in zip(preds, labels):
        for p_i, l_i in zip(p_row, l_row):
            if l_i == -100:
                continue
            true_labels.append(l_i)
            true_preds.append(p_i)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_preds, average="macro", zero_division=0
    )
    acc = (np.array(true_labels) == np.array(true_preds)).mean()
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# ============================================
# 8) TrainingArguments (NO evaluation_strategy here)
# ============================================
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/clinbert_lab_ner_option3",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    logging_steps=50,
    save_steps=200,
    fp16=False,      # CPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ============================================
# 9) Train + evaluate
# ============================================
train_output = trainer.train()
print(train_output)

eval_results = trainer.evaluate()
print(eval_results)

# ============================================
# 10) Save model locally
# ============================================
save_dir = "/content/drive/MyDrive/clinbert_lab_ner_option3"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3147465108.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.401600
100,0.024800
150,0.011600
200,0.007900
250,0.008000
300,0.005300
350,0.005200
400,0.003300


TrainOutput(global_step=400, training_loss=0.058458773531019685, metrics={'train_runtime': 4528.0325, 'train_samples_per_second': 0.353, 'train_steps_per_second': 0.088, 'total_flos': 209050634649600.0, 'train_loss': 0.058458773531019685, 'epoch': 1.0})


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.0036212464328855276, 'eval_accuracy': 0.9994355103380375, 'eval_precision': 0.8119434943615654, 'eval_recall': 0.8097205332513221, 'eval_f1': 0.8097605748229344, 'eval_runtime': 329.106, 'eval_samples_per_second': 1.215, 'eval_steps_per_second': 0.304, 'epoch': 1.0}
Saved to: /content/drive/MyDrive/clinbert_lab_ner_option3


In [22]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
